In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass
import math

# 1. CONFIG
@dataclass
class Config:
    vocab_size: int = 50257
    block_size: int = 2048
    n_layer: int = 12
    n_head: int = 12
    n_kv_head: int = 4
    n_embd: int = 768
    rope_theta: float = 500000.0
    norm_eps: float = 1e-6

# 2. RMSNorm
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.scale = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        return x * self.scale / (x.pow(2).mean(-1, keepdim=True) + self.eps).sqrt()

# 3. RoPE precompute
def precompute_rope_freqs(dim, max_len, theta=500000.0):
    freqs = 1.0 / (theta ** (torch.arange(0, dim, 2)[:dim//2].float() / dim))
    t = torch.arange(max_len, dtype=torch.float32)
    freqs = torch.outer(t, freqs)
    return torch.cos(freqs), torch.sin(freqs)

# 4. RoPE apply
def apply_rotary_emb(q, k, cos, sin, position_ids):
    cos = cos[position_ids].unsqueeze(1)
    sin = sin[position_ids].unsqueeze(1)
    
    head_dim = q.shape[-1]
    q_real, q_imag = q[..., :head_dim//2], q[..., head_dim//2:]
    k_real, k_imag = k[..., :head_dim//2], k[..., head_dim//2:]
    
    q_rot = torch.cat((q_real * cos - q_imag * sin, q_real * sin + q_imag * cos), dim=-1)
    k_rot = torch.cat((k_real * cos - k_imag * sin, k_real * sin + k_imag * cos), dim=-1)
    return q_rot, k_rot

# 5. BLOCK (Chunked GQA Implementation to prevent OOM)
class AmadeusZeroBlock(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.n_head = config.n_head
        self.n_kv_head = config.n_kv_head
        self.n_embd = config.n_embd
        self.head_dim = config.n_embd // config.n_head
        
        assert self.n_head % self.n_kv_head == 0
        self.num_key_value_groups = self.n_head // self.n_kv_head
        
        hidden_dim = int(8 * config.n_embd / 3)
        hidden_dim = ((hidden_dim + 255) // 256) * 256
        
        self.ln_1 = RMSNorm(config.n_embd, eps=config.norm_eps)
        self.ln_2 = RMSNorm(config.n_embd, eps=config.norm_eps)
        
        self.q_size = self.n_head * self.head_dim
        self.kv_size = self.n_kv_head * self.head_dim
        self.c_attn = nn.Linear(config.n_embd, self.q_size + 2 * self.kv_size, bias=False)
        self.c_proj = nn.Linear(config.n_embd, config.n_embd, bias=False)
        
        self.mlp = nn.ModuleDict({
            'gate_proj': nn.Linear(config.n_embd, hidden_dim, bias=False),
            'up_proj': nn.Linear(config.n_embd, hidden_dim, bias=False),
            'down_proj': nn.Linear(hidden_dim, config.n_embd, bias=False),
        })

    def forward(self, x, cos, sin, position_ids):
        x = x + self._attn_block(self.ln_1(x), cos, sin, position_ids)
        x = x + self._mlp_block(self.ln_2(x))
        return x

    def _attn_block(self, x, cos, sin, position_ids):
        B, T, C = x.size()
        
        qkv = self.c_attn(x)
        q, k, v = qkv.split([self.q_size, self.kv_size, self.kv_size], dim=2)
        
        q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_kv_head, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_kv_head, self.head_dim).transpose(1, 2)
        
        q, k = apply_rotary_emb(q, k, cos, sin, position_ids)

        # --------------------------------------------------------
        # THE FIX: Chunked GQA Math
        # We loop through the 4 KV heads sequentially. 
        # By using .expand() inside the loop, we map the memory 
        # without ever duplicating it physically.
        # --------------------------------------------------------
        y_chunks = []
        for i in range(self.n_kv_head):
            # Grab 3 Query heads
            q_chunk = q[:, i * self.num_key_value_groups : (i + 1) * self.num_key_value_groups, :, :]
            
            # Grab 1 KV head and expand it to match the 3 Query heads (zero memory copy)
            k_chunk = k[:, i : i + 1, :, :].expand(-1, self.num_key_value_groups, -1, -1)
            v_chunk = v[:, i : i + 1, :, :].expand(-1, self.num_key_value_groups, -1, -1)
            
            # Run SDPA on this small fraction of the data
            y_chunk = F.scaled_dot_product_attention(q_chunk, k_chunk, v_chunk, is_causal=True)
            y_chunks.append(y_chunk)

        # Recombine the 4 chunks back into a single 12-head tensor
        y = torch.cat(y_chunks, dim=1)
        # --------------------------------------------------------

        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.c_proj(y)

    def _mlp_block(self, x):
        gate = F.silu(self.mlp.gate_proj(x))
        up = self.mlp.up_proj(x)
        return self.mlp.down_proj(gate * up)

# 6. MAIN MODEL
class AmadeusZeroLM(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config

        self.transformer = nn.ModuleDict({
            'wte': nn.Embedding(config.vocab_size, config.n_embd),
            'h': nn.ModuleList([AmadeusZeroBlock(config) for _ in range(config.n_layer)]),
            'ln_f': RMSNorm(config.n_embd, eps=config.norm_eps),
        })
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)
        self.lm_head.weight = self.transformer.wte.weight
        
        dim = config.n_embd // config.n_head
        max_len = config.block_size * 2
        cos, sin = precompute_rope_freqs(dim, max_len, theta=config.rope_theta)
        self.register_buffer("cos", cos)
        self.register_buffer("sin", sin)
        
        self.apply(self._init_weights)

    def _init_weights(self, module):
        std = 0.02
        if isinstance(module, nn.Linear):
            if module.weight.size(0) == self.config.n_embd:
                std = 0.02 / math.sqrt(2 * self.config.n_layer)
            torch.nn.init.normal_(module.weight, mean=0.0, std=std)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None, position_ids=None):
        _, t = idx.size()
        assert t <= self.config.block_size
        
        if position_ids is None:
            position_ids = torch.arange(t, dtype=torch.long, device=idx.device).unsqueeze(0)
            
        tok_emb = self.transformer.wte(idx)
        
        x = tok_emb
        for block in self.transformer.h:
            x = block(x, self.cos, self.sin, position_ids)
        
        x = self.transformer.ln_f(x)
        logits = self.lm_head(x)
        
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss

In [2]:
import tiktoken
tokenizer = tiktoken.get_encoding("gpt2")

print(f"Vocabulary size: {tokenizer.n_vocab:,}")
print(f"Special tokens: {tokenizer.special_tokens_set}")

Vocabulary size: 50,257
Special tokens: {'<|endoftext|>'}


In [3]:
print(tokenizer.encode("<|endoftext|>", allowed_special="all"))

[50256]


In [4]:
conf = Config()
model = AmadeusZeroLM(conf)
# PENTING: Wrap DataParallel DULU baru pindahin ke .to(device)
if torch.cuda.device_count() > 1:
    print(f"Aktifkan {torch.cuda.device_count()} GPU T4!")
    model = nn.DataParallel(model)

device = torch.device("cuda")
model.to(device)

Aktifkan 2 GPU T4!


DataParallel(
  (module): AmadeusZeroLM(
    (transformer): ModuleDict(
      (wte): Embedding(50257, 768)
      (h): ModuleList(
        (0-11): 12 x AmadeusZeroBlock(
          (ln_1): RMSNorm()
          (ln_2): RMSNorm()
          (c_attn): Linear(in_features=768, out_features=1280, bias=False)
          (c_proj): Linear(in_features=768, out_features=768, bias=False)
          (mlp): ModuleDict(
            (gate_proj): Linear(in_features=768, out_features=2048, bias=False)
            (up_proj): Linear(in_features=768, out_features=2048, bias=False)
            (down_proj): Linear(in_features=2048, out_features=768, bias=False)
          )
        )
      )
      (ln_f): RMSNorm()
    )
    (lm_head): Linear(in_features=768, out_features=50257, bias=False)
  )
)

In [5]:
for name, param in model.named_parameters():
    if param.requires_grad:
        print(f"{name:20} | Shape: {str(param.shape):20} | Params: {param.numel():,}")

module.transformer.wte.weight | Shape: torch.Size([50257, 768]) | Params: 38,597,376
module.transformer.h.0.ln_1.scale | Shape: torch.Size([768])    | Params: 768
module.transformer.h.0.ln_2.scale | Shape: torch.Size([768])    | Params: 768
module.transformer.h.0.c_attn.weight | Shape: torch.Size([1280, 768]) | Params: 983,040
module.transformer.h.0.c_proj.weight | Shape: torch.Size([768, 768]) | Params: 589,824
module.transformer.h.0.mlp.gate_proj.weight | Shape: torch.Size([2048, 768]) | Params: 1,572,864
module.transformer.h.0.mlp.up_proj.weight | Shape: torch.Size([2048, 768]) | Params: 1,572,864
module.transformer.h.0.mlp.down_proj.weight | Shape: torch.Size([768, 2048]) | Params: 1,572,864
module.transformer.h.1.ln_1.scale | Shape: torch.Size([768])    | Params: 768
module.transformer.h.1.ln_2.scale | Shape: torch.Size([768])    | Params: 768
module.transformer.h.1.c_attn.weight | Shape: torch.Size([1280, 768]) | Params: 983,040
module.transformer.h.1.c_proj.weight | Shape: torch

In [6]:
def count_parameters(model):
    # Sum semua elemen di tiap parameter yang butuh gradien
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f'AmadeusZeroLM Parameter: {count_parameters(model):,}')

AmadeusZeroLM Parameter: 114,114,048


In [7]:
conf = Config()
assert isinstance(conf.vocab_size, int), "CONFIG Broken!"
print("Config OK:", conf.vocab_size)  

Config OK: 50257


In [8]:
import os
from kaggle_secrets import UserSecretsClient

# Fetch the secret key from Kaggle Secrets
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("HUGGINGFACE_KEY")

# Set the environment variable for Hugging Face integration
os.environ["HF_TOKEN"] = secret_value_0

In [9]:
from huggingface_hub import hf_hub_download
import pandas as pd
# 2. Download the correct parquet file from the repository
repo_id = "karpathy/tinystories-gpt4-clean"
filename = "tinystories_gpt4_clean.parquet"

print(f"Downloading {filename} from {repo_id}...")
downloaded_path = hf_hub_download(
    repo_id=repo_id,
    filename=filename,
    repo_type="dataset"
)

# 3. Process the Parquet file using pandas
# Reading the binary format correctly into a structured DataFrame
df = pd.read_parquet(downloaded_path)

# Extracting the text column (assuming 'text' or 'story' is the column name)
# If the column name is different, we grab the first column automatically
text_column = df.columns[0]
stories = df[text_column].tolist()

print("First story (300 chars):\n")
story = stories[0]
print(story.strip()[:300], "...")

print(f"\nTotal number of stories: {len(stories):,}")

tinystories_gpt4_clean.parquet:   0%|          | 0.00/673M [00:00<?, ?B/s]

First story (300 chars):

Once there was a little boy named Jack. He was only three years old and had lots of things he wanted to do. One day he saw something very special in the park - a big wheel! It was big and bright and looked very inviting.
Jack wanted to get on the wheel, so he ran to it. He put his hands on it and ge ...

Total number of stories: 2,732,634


In [10]:
import os
import math
import torch
from torch.utils.data import Dataset, DataLoader

# 1. Custom Dataset for tokenizing the stories
class StoryDataset(Dataset):
    def __init__(self, texts, tokenizer, block_size):
        self.texts = texts
        self.tokenizer = tokenizer
        self.block_size = block_size
        # GPT-2 uses 50256 as the <|endoftext|> token, which we can use for padding
        self.pad_token_id = 50256 

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        tokens = self.tokenizer.encode(text, allowed_special="all")
        
        # We need block_size + 1 tokens total (to create shifted x and y)
        max_len = self.block_size + 1
        
        if len(tokens) < max_len:
            # Pad sequences that are too short
            tokens.extend([self.pad_token_id] * (max_len - len(tokens)))
        else:
            # Truncate sequences that are too long
            tokens = tokens[:max_len]
            
        # x is the input sequence, y is the target sequence shifted by 1
        x = torch.tensor(tokens[:-1], dtype=torch.long)
        y = torch.tensor(tokens[1:], dtype=torch.long)
        
        return x, y

# Create the datasets and dataloaders
train_dataset = StoryDataset(stories[:10000], tokenizer, conf.block_size)
train_loader = DataLoader(train_dataset, batch_size=1, shuffle=True, num_workers=2)
total_steps = len(train_loader)

val_dataset = StoryDataset(stories[2600000:], tokenizer, conf.block_size)
val_loader = DataLoader(val_dataset, batch_size=1, shuffle=True, num_workers=2)

print(f"Total stories in dataset: {len(train_dataset):,}")
print(f"Batch size: {train_loader.batch_size}")
print(f"Total steps per epoch actually: {total_steps:,}")
print(f"Total stories in validation set: {len(val_dataset):,}")

Total stories in dataset: 10,000
Batch size: 8
Total steps per epoch actually: 1,250
Total stories in validation set: 132,634


In [12]:
# 1. Setup the Optimizer ONLY
epochs = 1  # Make sure you increase this if you want to run the next epoch!
learning_rate = 3e-4
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

# 2. Checkpoint Settings
start_epoch = 0
start_step = 0

print("Checking for checkpoint on Hugging Face...")
try:
    checkpoint_path = hf_hub_download(
        repo_id="RinKana/AmadeusZero-114M", 
        filename="kaggle/working/amadeus_final.pt"
    )
    
    print(f"Found checkpoint at {checkpoint_path}. Loading...")
    
    checkpoint = torch.load(checkpoint_path, map_location='cpu')
    model.load_state_dict(checkpoint['model_state_dict'])
    
    # Load optimizer state
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    
    # Force the optimizer to use the correct learning rate
    for param_group in optimizer.param_groups:
        param_group['lr'] = learning_rate
        param_group['initial_lr'] = learning_rate
    
    start_epoch = checkpoint['epoch']
    saved_step = checkpoint['step']
    
    # SAFELY HANDLE DYNAMIC DATASET SIZES
    # Instead of relying on math that changes when the dataset changes,
    # we check if we are starting a brand new epoch.
    
    # If you explicitly want to start Epoch 1 (or the next epoch), set this to True
    STARTING_NEW_EPOCH = True 
    
    if STARTING_NEW_EPOCH:
        print("Initializing new epoch with dynamic dataset size. Resetting steps.")
        start_epoch += 1     # Move to the next epoch
        start_step = 0       # Start from the very first batch
    else:
        # Only use this if resuming an interrupted run on the EXACT SAME dataset
        start_step = saved_step + 1 
        
    print(f"Successfully resumed. Targeting Epoch {start_epoch}, starting at Step {start_step}")
    
except Exception as e:
    print(f"Download failed: {e}")
    print("No checkpoint found. Starting training from scratch.")

# 3. Create a FRESH scheduler tailored to the new dataset size
# Do NOT load the scheduler_state_dict if the dataset size changed.
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=total_steps)
# Function to estimate validation loss quickly without computing gradients
@torch.no_grad()
def estimate_loss(model, dataloader, eval_iters=50):
    model.eval()
    losses = torch.zeros(eval_iters)
    for k, (x, y) in enumerate(dataloader):
        if k >= eval_iters:
            break
        x, y = x.to(device), y.to(device)
        
        # Validation must also run in mixed precision to prevent FP32 OOM
        with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
            logits, loss = model(x, targets=y)
            
        losses[k] = loss.mean().item()
        
        # Destroy the validation graph tensors immediately to prevent loop buildup
        del logits, loss
        
    model.train()
    return losses.mean().item()

# 5. Training Loop
model.train()

print("Starting training loop...")
for epoch in range(start_epoch, epochs):
    for step, (x, y) in enumerate(train_loader):
        if epoch == start_epoch and step < start_step:
            continue
            
        x, y = x.to(device), y.to(device)
        
        # Forward pass in mixed precision to allow native SDPA GQA
        with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
            logits, loss = model(x, targets=y)
        
        loss = loss.mean()
        perplexity = torch.exp(loss)
        
        # Extract scalar numbers before deleting tensors
        train_loss_val = loss.item()
        train_ppl_val = perplexity.item()
        
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        
        # THE FIX: Annihilate the massive computational graph tensors
        del logits, loss, perplexity
        
        if step % 1000 == 0:
            # THE FIX: Defragment the VRAM before validation runs
            torch.cuda.empty_cache()
            
            val_loss = estimate_loss(model, val_loader, eval_iters=50)
            val_perplexity = math.exp(val_loss)
            current_lr = scheduler.get_last_lr()[0]
            
            print(f"Epoch: {epoch} | Step: {step} | LR: {current_lr:.6f} | Train Loss: {train_loss_val:.4f} | Val Loss: {val_loss:.4f} | Val PPL: {val_perplexity:.4f} | Perplexity: {train_ppl_val:.4f}") 
            
        is_last_step = step == len(train_loader) - 1
        
        if (step > 0 and step % 10000 == 0) or is_last_step:
            print(f"Saving checkpoint at Epoch {epoch}, Step {step}...")
            
            if step % 1000 != 0:
                torch.cuda.empty_cache()
                val_loss = estimate_loss(model, val_loader, eval_iters=50)
                val_perplexity = math.exp(val_loss)

            torch.save({
                'epoch': epoch,
                'step': step,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'scheduler_state_dict': scheduler.state_dict(),
                'loss': train_loss_val,
                'perplexity': train_ppl_val,
                'val_loss': val_loss,
                'val_perplexity': val_perplexity,
            }, "amadeus_final.pt")

print("Training complete. Saving final checkpoint...")

Checking for checkpoint on Hugging Face...
Download failed: 404 Client Error. (Request ID: Root=1-6a27ef3f-798532f949685c82338902ac;d73f2721-8eba-4eb4-9e2a-e6754f927841)

Repository Not Found for url: https://huggingface.co/RinKana/AmadeusZero-114M/resolve/main/kaggle/working/amadeus_final.pt.
Please make sure you specified the correct `repo_id` and `repo_type`.
If you are trying to access a private or gated repo, make sure you are authenticated and your token has the required permissions.
For more details, see https://huggingface.co/docs/huggingface_hub/authentication
No checkpoint found. Starting training from scratch.
Starting training loop...


RuntimeError: Caught RuntimeError in replica 0 on device 0.
Original Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/parallel_apply.py", line 103, in _worker
    output = module(*input, **kwargs)
             ^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py", line 1776, in _wrapped_call_impl
    return self._call_impl(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py", line 1787, in _call_impl
    return forward_call(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_58/2588662651.py", line 171, in forward
    x = block(x, self.cos, self.sin, position_ids)
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py", line 1776, in _wrapped_call_impl
    return self._call_impl(*args, **kwargs)
           ^